# StockSense - Data Analysis & Forecasting Notebook
---
This notebook documents the complete data science workflow for StockSense:
1. **Data Extraction** from ERPNext (MariaDB)
2. **Exploratory Data Analysis (EDA)** - understanding stock patterns
3. **Demand Forecasting** - predicting future stock needs
4. **Anomaly Detection** - identifying suspicious movements
5. **Results Export** for Power BI integration

## 1. Setup & Database Connection

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
from pathlib import Path

# Style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)

# Load credentials
env_path = Path('../.env')
load_dotenv(dotenv_path=env_path)

# Connect to ERPNext DB
engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST', 'localhost')}:{os.getenv('DB_PORT', '3306')}"
    f"/{os.getenv('DB_NAME')}"
)
print('Connected to ERPNext database successfully!')

## 2. Data Extraction

In [ ]:
# Current stock snapshot
df_stock = pd.read_sql(text("""
    SELECT b.item_code, i.item_name, i.item_group, b.warehouse,
           b.actual_qty, b.projected_qty, b.valuation_rate,
           (b.actual_qty * b.valuation_rate) AS stock_value
    FROM `tabBin` b
    LEFT JOIN `tabItem` i ON b.item_code = i.name
    WHERE i.is_stock_item = 1
"""), engine)

print(f'Stock records: {len(df_stock)}')
df_stock.head(10)

In [ ]:
# Stock ledger (movement history)
df_ledger = pd.read_sql(text("""
    SELECT item_code, warehouse, posting_date, actual_qty,
           qty_after_transaction, voucher_type, voucher_no, valuation_rate
    FROM `tabStock Ledger Entry`
    ORDER BY posting_date DESC
"""), engine)

df_ledger['posting_date'] = pd.to_datetime(df_ledger['posting_date'])
print(f'Ledger entries: {len(df_ledger)}')
df_ledger.head(10)

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Stock value distribution by item
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Chart 1: Stock Quantity by Item
stock_by_item = df_stock.groupby('item_name')['actual_qty'].sum().sort_values(ascending=True)
stock_by_item.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Current Stock Quantity by Item', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Quantity')

# Chart 2: Stock Value by Item
value_by_item = df_stock.groupby('item_name')['stock_value'].sum().sort_values(ascending=True)
value_by_item.plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Stock Value (INR) by Item', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Value (INR)')

plt.tight_layout()
plt.savefig('../../data/eda_stock_overview.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Daily movement trends
daily_movements = df_ledger.groupby('posting_date')['actual_qty'].agg(['sum', 'count']).reset_index()
daily_movements.columns = ['date', 'net_movement', 'transaction_count']

fig, ax1 = plt.subplots(figsize=(14, 5))
ax1.bar(daily_movements['date'], daily_movements['transaction_count'], alpha=0.3, color='gray', label='Transactions')
ax2 = ax1.twinx()
ax2.plot(daily_movements['date'], daily_movements['net_movement'], color='red', linewidth=2, label='Net Movement')

ax1.set_xlabel('Date')
ax1.set_ylabel('Transaction Count', color='gray')
ax2.set_ylabel('Net Qty Movement', color='red')
plt.title('Daily Stock Movement Trends', fontsize=14, fontweight='bold')
fig.legend(loc='upper right', bbox_to_anchor=(0.95, 0.95))
plt.tight_layout()
plt.savefig('../../data/eda_daily_trends.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Demand Forecasting
Using Gradient Boosting to predict daily outflow demand per item.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

# Prepare daily outflow per item
outflow = df_ledger[df_ledger['actual_qty'] < 0].copy()
outflow['daily_outflow'] = outflow['actual_qty'].abs()
daily_demand = outflow.groupby(['item_code', 'posting_date'])['daily_outflow'].sum().reset_index()
daily_demand.columns = ['item_code', 'date', 'demand']
daily_demand['day_of_week'] = daily_demand['date'].dt.dayofweek
daily_demand['day_of_month'] = daily_demand['date'].dt.day

# Train per item
forecast_results = []
for item in daily_demand['item_code'].unique():
    item_data = daily_demand[daily_demand['item_code'] == item].sort_values('date')
    if len(item_data) < 5:
        continue
    
    X = item_data[['day_of_week', 'day_of_month']]
    y = item_data['demand']
    split = int(len(X) * 0.8)
    
    model = GradientBoostingRegressor(n_estimators=50, max_depth=3, random_state=42)
    model.fit(X[:split], y[:split])
    preds = model.predict(X[split:])
    mae = mean_absolute_error(y[split:], preds)
    
    forecast_results.append({
        'item_code': item,
        'avg_daily_forecast': round(np.mean(preds), 2),
        'mae': round(mae, 2)
    })

df_forecast = pd.DataFrame(forecast_results)
print('Demand Forecast Results:')
df_forecast

## 5. Anomaly Detection
Using Z-Score method to flag unusual stock movements.

In [ ]:
# Z-Score anomaly detection
stats = df_ledger.groupby('item_code')['actual_qty'].agg(['mean', 'std']).reset_index()
stats.columns = ['item_code', 'qty_mean', 'qty_std']
stats['qty_std'] = stats['qty_std'].replace(0, 1)

df_analysis = df_ledger.merge(stats, on='item_code', how='left')
df_analysis['z_score'] = (df_analysis['actual_qty'] - df_analysis['qty_mean']) / df_analysis['qty_std']
df_analysis['is_anomaly'] = df_analysis['z_score'].abs() > 2.5

anomalies = df_analysis[df_analysis['is_anomaly']]
print(f'Anomalies detected: {len(anomalies)} out of {len(df_analysis)} transactions')

if not anomalies.empty:
    anomalies[['item_code', 'posting_date', 'actual_qty', 'z_score', 'voucher_type']].head(20)
else:
    print('No anomalies found - all movements within normal range.')

In [ ]:
# Visualize anomalies
fig, ax = plt.subplots(figsize=(14, 6))

normal = df_analysis[~df_analysis['is_anomaly']]
anomaly = df_analysis[df_analysis['is_anomaly']]

ax.scatter(normal['posting_date'], normal['actual_qty'], alpha=0.3, s=20, color='steelblue', label='Normal')
ax.scatter(anomaly['posting_date'], anomaly['actual_qty'], alpha=0.8, s=80, color='red', marker='x', label='Anomaly')

ax.set_xlabel('Date')
ax.set_ylabel('Quantity')
ax.set_title('Stock Movements: Normal vs Anomalous', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../../data/anomaly_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Export Results
Save forecast and anomaly results as CSV for Power BI consumption.

In [ ]:
# Save forecast results
df_forecast.to_csv('../../data/demand_forecast_results.csv', index=False)
print('Forecast results saved to data/demand_forecast_results.csv')

# Save anomaly report
if not anomalies.empty:
    anomalies[['item_code', 'warehouse', 'posting_date', 'actual_qty', 'z_score', 'voucher_type', 'voucher_no']].to_csv(
        '../../data/anomaly_report.csv', index=False
    )
    print('Anomaly report saved to data/anomaly_report.csv')
else:
    print('No anomalies to export.')

print('\nAll results exported! Load these CSVs into Power BI using the M scripts.')